# 1) Imports & chargement

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer

In [2]:
df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')

# 2) Drop des colonnes avec 80+% de NaN et inutiles

In [3]:
valeurs_manquantes = df_ventes.isna().sum()/df_ventes.shape[0]
valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)

df_ventes_filtered = df_ventes.drop(valeurs_manquantes_list, axis = 1)

column_n6 = [ column for column in df_ventes_filtered.columns if "n6" in column]
df_ventes_filtered = df_ventes_filtered.drop(column_n6, axis = 1)

# 3) Premier nettoyage et conversion de float à int

In [4]:
df_ventes_filtered = df_ventes_filtered[df_ventes_filtered['typedebien'] != 'l']
df_ventes_filtered['typedebien'] = df_ventes_filtered['typedebien'].replace({'an': 'a', 'mn': 'm'})

valeurs_manq_resid_quanti = [col for col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]

col_float = df_ventes_filtered[valeurs_manq_resid_quanti].select_dtypes(include='float64').columns
for col in col_float:
    array_col = np.array(df_ventes_filtered[df_ventes_filtered[col].notna()][col]) 
    array_col_round = np.round(array_col)
    array_real_float = array_col[array_col != array_col_round]
    if len(array_real_float) == 0:
        df_ventes_filtered[col] = df_ventes_filtered[col].astype('Int64')

df_ventes_filtered["cave"] = df_ventes_filtered["cave"].astype('Int64')
df_ventes_filtered["ascenseur"] = df_ventes_filtered["ascenseur"].astype('Int64')
df_ventes_filtered["logement_neuf"] = df_ventes_filtered["logement_neuf"].replace({'n': False, 'o': True}).astype('Int64')

In [5]:
df_ventes = df_ventes_filtered
target = df_ventes['prix_bien']

# 4) Split train/test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df_ventes.drop(columns=['prix_bien']),
    target,
    test_size=0.2,
    random_state=42
)

# 5) Remplissage des colonnes quali

In [7]:
neighbours_columns = ["nb_pieces", "mapCoordonneesLatitude", "mapCoordonneesLongitude"]
type_column = 'typedebien'

def df_imputed(input_df: pd.DataFrame, test_df: pd.DataFrame, col_processed, default):
    imputer = KNNImputer()
    cols_backup = input_df.columns
    index_backup = input_df.index
    test_index = test_df.index
    if input_df.shape[0] > 0:
        imputed_data = imputer.fit_transform(input_df)
        if imputed_data.shape[1] == len(cols_backup):
            imputed_df = pd.DataFrame(data=imputed_data, columns=cols_backup, index=index_backup)
            if test_df.shape[0] > 0:
                test_data = imputer.transform(test_df)
                imputed_test_df = pd.DataFrame(data=test_data, columns=cols_backup, index=test_index)
            else:
                imputed_test_df = test_df
        else: 
            # Case when the input can not have been done
            cols_wo_processed = cols_backup.drop(col_processed)
            imputed_df = pd.DataFrame(data=imputed_data, columns=cols_wo_processed, index=index_backup)
            imputed_df[col_processed] = default
            imputed_test_df = pd.DataFrame(data=test_df[cols_wo_processed], columns=cols_wo_processed, index=test_index)
            imputed_test_df[col_processed] = default
            
    else:
        imputed_data = input_df
        imputed_test_df = test_df
        
    return imputed_df, imputed_test_df


valeurs_manq_resid_quanti = [col for  col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]

for col in valeurs_manq_resid_quanti:
    print(f"Processing column {col}")
    # On se focalise sur la colonne et nos colonnes voisines
    df_extract = X_train[[type_column] + [col] + neighbours_columns]
    df_test_extract = X_test[[type_column] + [col] + neighbours_columns]
    default_value = X_train[col].mean()
    # On distingue par type de bien
    df_extract_app = df_extract[df_extract[type_column] =='a'][[col] + neighbours_columns]
    df_extract_mais = df_extract[df_extract[type_column] =='m'][[col] + neighbours_columns]
    df_test_extract_app = df_test_extract[df_test_extract[type_column] =='a'][[col] + neighbours_columns]
    df_test_extract_mais = df_test_extract[df_test_extract[type_column] =='m'][[col] + neighbours_columns]
    
    df_filled_app, df_test_filled_app = df_imputed(df_extract_app, df_test_extract_app, col, default_value)
    df_filled_mais, df_test_filled_mais = df_imputed(df_extract_mais, df_test_extract_mais, col, default_value)
    df_filled = pd.concat([df_filled_app, df_filled_mais])
    df_test_filled = pd.concat([df_test_filled_app, df_test_filled_mais])

    if X_train[col].dtypes in ['int64', 'Int64']:
        X_train[col] = df_filled[col].apply(lambda x: round(x)).astype('int64')
        X_test[col] = df_test_filled[col].apply(lambda x: round(x)).astype('int64')
    else:
        X_train[col] = df_filled[col]
        X_test[col] = df_test_filled[col]

Processing column surface_terrain
Processing column dpeC
Processing column nb_etages
Processing column places_parking
Processing column cave
Processing column annee_construction
Processing column nb_toilettes
Processing column ascenseur
Processing column nb_logements_copro
Processing column charges_copro
Processing column logement_neuf
Processing column duree_int
Processing column loyer_m2_median_n7
Processing column nb_log_n7
Processing column taux_rendement_n7


# 6) Remplissage des colonnes quanti

In [8]:
def fit_and_or_transform(df_train, df_test = None):
    if df_test is None:
        input_df = df_train
    else:
        input_df = df_test
    valeurs_manq_resid_quali = [col for  col in input_df.select_dtypes(include='object').columns if input_df[col].isna().sum() > 0]
    
    # On va itérer sur chaque ligne avec une valeur manquante
    index_na = list(input_df[input_df[valeurs_manq_resid_quali].isna().any(axis = 1)].index)
    len_index_na = len(index_na)
    print(f"Going to process {len_index_na} records")
    i = 1
    # Valeurs par défaut
    default_mode = {col_quali: df_train[col_quali].mode()[0] for col_quali in valeurs_manq_resid_quali}
    for index in index_na:
        if i % 100 == 0:
            print(f"\rprocessing index {i}/{len_index_na}", end = "")
        i += 1
        latitude = input_df.loc[index, 'mapCoordonneesLatitude']
        longitude = input_df.loc[index, 'mapCoordonneesLongitude']
        nb_piece = input_df.loc[index, 'nb_pieces']
        type_bien = input_df.loc[index, 'typedebien']
    
        # On selectionne les enregistrements du dataframe de train même type avec le même nombre de pièces
        neighbours = df_train[(df_train['nb_pieces'] == nb_piece) & (df_train['typedebien'] == type_bien)]
        # On calcule une fois le vecteur de distance pour l'ensemble des voisins (colonnes quanti à na ou non)
        neighbours_distance = (latitude - neighbours['mapCoordonneesLatitude'])**2 +(longitude - neighbours['mapCoordonneesLongitude'])**2  
        # On itère sur les variables quali manquantes de l'enregistrement
        var_col_quali = input_df.loc[index][valeurs_manq_resid_quali].isna()
        for col_quali in var_col_quali[var_col_quali].index:
            ## On prend les 10 plus proches voisins n'ayant pas la variable à Na
            neighbours_indexes = neighbours_distance[neighbours[col_quali].notna()].sort_values().iloc[:10].index
            if len(neighbours_indexes) > 0:
                input_df.loc[index, col_quali] = neighbours.loc[neighbours_indexes][col_quali].mode()[0]
            else:
                print(f"\nInfo: no nearest neighbours found for index {index} and col {col_quali}")
                input_df.loc[index, col_quali] = default_mode[col_quali]
    print("\n")

def fit_transform(df_train):
    fit_and_or_transform(df_train)

def transform(df_train, df_test):
    fit_and_or_transform(df_train, df_test)

print("Processing X_train")
fit_transform(X_train)
print("Processing X_test")
transform(X_train, X_test)

Processing X_train
Going to process 17978 records
processing index 800/17978
Info: no nearest neighbours found for index ag681124-329355546 and col chauffage_systeme
processing index 1100/17978
Info: no nearest neighbours found for index ag681237-323941230 and col chauffage_systeme
processing index 1400/17978
Info: no nearest neighbours found for index ag671791-374042365 and col chauffage_systeme

Info: no nearest neighbours found for index ag671791-374042365 and col chauffage_mode
processing index 2800/17978
Info: no nearest neighbours found for index immo-facile-3797119 and col chauffage_systeme
processing index 4000/17978
Info: no nearest neighbours found for index hektor-ghisimmobilier-1876 and col chauffage_energie

Info: no nearest neighbours found for index hektor-ghisimmobilier-1876 and col chauffage_mode
processing index 10000/17978
Info: no nearest neighbours found for index ag681404-326620851 and col ges_class

Info: no nearest neighbours found for index ag681404-326620851 a

# 7) Fonction de nettoyage des outiliers (IQR)

In [10]:
# Suppression de certains champs dont nous ne connaissons pas la définition
X_train.drop(columns=['duree_int', 'loyer_m2_median_n7', 'nb_log_n7', 'taux_rendement_n7'], axis=1, inplace=True)

In [11]:
X_train_clean = X_train

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()

index_to_drop = []

for col in numeric_cols:
    q1 = max(X_train[col].quantile(0.1), 0)
    q3 = max(X_train[col].quantile(0.9), 0)
    iqr = q3 - q1
    lower = max(q1 - 2 * iqr, 0)
    upper = max(q3 + 2 * iqr, 0)
    temp_index_to_drop = X_train[(X_train[col] < lower) | (X_train[col] > upper)].index.to_list()
    index_to_drop.extend(temp_index_to_drop)
    print("Suppression des lignes où ", col , " < ", lower, " ou ", col, " > ", upper, " (", len(temp_index_to_drop), " rows)")

X_train_clean = X_train_clean.drop(index_to_drop)

nb_rows_avant = X_train.shape[0]
nb_rows_apres = X_train_clean.shape[0]
nb_rows_suppr = nb_rows_avant - nb_rows_apres

print("\ndataframe initial : ", nb_rows_avant, " rows.")
print("dataframe nettoyé avec iqr : ", nb_rows_apres, " rows.")
print("Proportion conservé : ", 100 * np.round(nb_rows_apres/nb_rows_avant, 5), "%")
print("Nombre d'observations supprimées : ", nb_rows_suppr)

X_train = X_train_clean

Suppression des lignes où  etage  <  0  ou  etage  >  6.0  ( 272  rows)
Suppression des lignes où  surface  <  0  ou  surface  >  421.0  ( 105  rows)
Suppression des lignes où  surface_terrain  <  0  ou  surface_terrain  >  2850.0  ( 351  rows)
Suppression des lignes où  nb_pieces  <  0  ou  nb_pieces  >  17.0  ( 47  rows)
Suppression des lignes où  mensualiteFinance  <  0.0  ou  mensualiteFinance  >  0.0  ( 331  rows)
Suppression des lignes où  balcon  <  0  ou  balcon  >  3.0  ( 3  rows)
Suppression des lignes où  eau  <  0  ou  eau  >  3.0  ( 16  rows)
Suppression des lignes où  bain  <  0  ou  bain  >  3.0  ( 89  rows)
Suppression des lignes où  dpeC  <  0  ou  dpeC  >  745.8  ( 13  rows)
Suppression des lignes où  mapCoordonneesLatitude  <  46.595870000000005  ou  mapCoordonneesLatitude  >  49.07131999999999  ( 0  rows)
Suppression des lignes où  mapCoordonneesLongitude  <  6.43378  ou  mapCoordonneesLongitude  >  8.263480000000001  ( 0  rows)
Suppression des lignes où  nb_etages 

# 8) Normalisation des données GPS (fit sur train)

In [12]:
scaler_lat = StandardScaler()
scaler_lon = StandardScaler()

X_train['Latitude_scaled']  = scaler_lat.fit_transform(X_train[['mapCoordonneesLatitude']])
X_train['Longitude_scaled'] = scaler_lon.fit_transform(X_train[['mapCoordonneesLongitude']])

X_test['Latitude_scaled']  = scaler_lat.transform(X_test[['mapCoordonneesLatitude']])
X_test['Longitude_scaled'] = scaler_lon.transform(X_test[['mapCoordonneesLongitude']])

# 9) Target Encoding (fit sur train)

In [13]:
cols_te = ['INSEE_COM', 'typedebien_lite', 'nb_pieces']

mean_price_by_combo = (
    X_train.assign(prix_bien=y_train)
    .groupby(cols_te, as_index=False)['prix_bien']
    .mean()
    .rename(columns={'prix_bien': 'prix_bien_target_encoding'})
)

X_train = X_train.merge(mean_price_by_combo, on=cols_te, how='left')
X_test  = X_test.merge(mean_price_by_combo, on=cols_te, how='left')

In [14]:
scaler_te = StandardScaler()
X_train['prix_bien_target_encoding_scaled'] = scaler_te.fit_transform(
    X_train[['prix_bien_target_encoding']]
)
X_test['prix_bien_target_encoding_scaled'] = scaler_te.transform(
    X_test[['prix_bien_target_encoding']]
)

# 10) Extraction année + OHE (fit sur train)

In [15]:
X_train['date'] = pd.to_datetime(X_train['date'])
X_test['date'] = pd.to_datetime(X_test['date'])

X_train['annee'] = X_train['date'].dt.year
X_test['annee'] = X_test['date'].dt.year

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
annee_train = ohe.fit_transform(X_train[['annee']])
annee_test  = ohe.transform(X_test[['annee']])

annee_cols = ohe.get_feature_names_out(['annee'])

X_train = pd.concat([X_train, pd.DataFrame(annee_train, columns=annee_cols, index=X_train.index)], axis=1)
X_test  = pd.concat([X_test,  pd.DataFrame(annee_test,  columns=annee_cols, index=X_test.index)], axis=1)

# 11) Parsing exposition / chauffage / DPE / GES

In [16]:
def parse_exposition(df):
    df = df.copy()
    df['exposition_clean'] = (
        df['exposition'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    df['expo_nord']  = df['exposition_clean'].str.contains(r'\bnord\b',  na=False).astype(int)
    df['expo_sud']   = df['exposition_clean'].str.contains(r'\bsud\b',   na=False).astype(int)
    df['expo_est']   = df['exposition_clean'].str.contains(r'\best\b',   na=False).astype(int)
    df['expo_ouest'] = df['exposition_clean'].str.contains(r'\bouest\b', na=False).astype(int)
    df['expo_inconnue'] = df['exposition_clean'].str.contains(r'0|nan', na=False).astype(int)
    return df

In [17]:
def parse_chauffage_systeme(df):
    df = df.copy()
    df['chauffage_systeme_clean'] = (
        df['chauffage_systeme'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['chauf_radiateur']  = df['chauffage_systeme_clean'].str.contains(r'\bradiateur\b', na=False).astype(int)
    df['chauf_sol']        = df['chauffage_systeme_clean'].str.contains(r'\bsol\b', na=False).astype(int)
    df['chauf_convecteur'] = df['chauffage_systeme_clean'].str.contains(r'\bconvecteur\b', na=False).astype(int)
    df['chauf_poele_bois'] = df['chauffage_systeme_clean'].str.contains(r'poêle|poele', na=False).astype(int)
    df['chauf_pac']        = df['chauffage_systeme_clean'].str.contains(r'pompe à chaleur|pac', na=False).astype(int)
    df['chauf_clim_rev']   = df['chauffage_systeme_clean'].str.contains(r'climatisation', na=False).astype(int)
    df['chauf_cheminee']   = df['chauffage_systeme_clean'].str.contains(r'cheminée|cheminee', na=False).astype(int)
    df['chauf_inconnu']    = df['chauffage_systeme_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [18]:
def parse_chauffage_energie(df):
    df = df.copy()
    df['chauffage_energie_clean'] = (
        df['chauffage_energie'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['energie_gaz']   = df['chauffage_energie_clean'].str.contains(r'\bgaz\b', na=False).astype(int)
    df['energie_elec']  = df['chauffage_energie_clean'].str.contains(r'électrique|electrique', na=False).astype(int)
    df['energie_fioul'] = df['chauffage_energie_clean'].str.contains(r'\bfioul\b', na=False).astype(int)
    df['energie_bois']  = df['chauffage_energie_clean'].str.contains(r'\bbois\b', na=False).astype(int)
    df['energie_inconnue'] = df['chauffage_energie_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [19]:
def parse_chauffage_mode(df):
    df = df.copy()
    df['chauffage_mode_clean'] = (
        df['chauffage_mode'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['chauffage_mode_individuel'] = df['chauffage_mode_clean'].str.contains(r'\bindividuel\b', na=False).astype(int)
    df['chauffage_mode_collectif']  = df['chauffage_mode_clean'].str.contains(r'\bcollectif\b', na=False).astype(int)
    df['chauffage_mode_central']    = df['chauffage_mode_clean'].str.contains(r'\bcentral\b', na=False).astype(int)
    df['chauffage_mode_inconnu']    = df['chauffage_mode_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [20]:
def parse_dpe(df):
    df = df.copy()
    df['dpe_A'] = df['dpeL'].str.contains(r'A', na=False).astype(int)
    df['dpe_B'] = df['dpeL'].str.contains(r'B', na=False).astype(int)
    df['dpe_C'] = df['dpeL'].str.contains(r'C', na=False).astype(int)
    df['dpe_D'] = df['dpeL'].str.contains(r'D', na=False).astype(int)
    df['dpe_E'] = df['dpeL'].str.contains(r'E', na=False).astype(int)
    df['dpe_F'] = df['dpeL'].str.contains(r'F', na=False).astype(int)
    df['dpe_G'] = df['dpeL'].str.contains(r'G', na=False).astype(int)
    df['dpe_inconnu'] = (~df['dpeL'].isin(list("ABCDEFG"))).astype(int)
    return df

In [21]:
def parse_ges(df):
    df = df.copy()
    df['ges_A'] = df['ges_class'].str.contains(r'A', na=False).astype(int)
    df['ges_B'] = df['ges_class'].str.contains(r'B', na=False).astype(int)
    df['ges_C'] = df['ges_class'].str.contains(r'C', na=False).astype(int)
    df['ges_D'] = df['ges_class'].str.contains(r'D', na=False).astype(int)
    df['ges_E'] = df['ges_class'].str.contains(r'E', na=False).astype(int)
    df['ges_F'] = df['ges_class'].str.contains(r'F', na=False).astype(int)
    df['ges_G'] = df['ges_class'].str.contains(r'G', na=False).astype(int)
    df['ges_inconnu'] = (~df['ges_class'].isin(list("ABCDEFG"))).astype(int)
    return df

In [22]:
X_train = parse_exposition(X_train)
X_test  = parse_exposition(X_test)

X_train = parse_chauffage_systeme(X_train)
X_test  = parse_chauffage_systeme(X_test)

X_train = parse_chauffage_energie(X_train)
X_test  = parse_chauffage_energie(X_test)

X_train = parse_dpe(X_train)
X_test  = parse_dpe(X_test)

X_train = parse_ges(X_train)
X_test  = parse_ges(X_test)

X_train = parse_chauffage_mode(X_train)
X_test  = parse_chauffage_mode(X_test)

# 12) OHE sur le reste des variables catégorielles

In [24]:
# --- Liste des colonnes à exclure (déjà parsées ou inutiles)
exclude_cols = [
    'mapCoordonneesLatitude', 'mapCoordonneesLongitude',
    'date', 'annee',
    'exposition', 'exposition_clean',
    'chauffage_systeme', 'chauffage_systeme_clean',
    'chauffage_energie', 'chauffage_energie_clean',
    'dpeL', 'ges_class',
    'chauffage_mode', 'chauffage_mode_clean'
]

# --- Sélection des colonnes catégorielles restantes
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
cat_cols = [c for c in cat_cols if c not in exclude_cols]

print("Colonnes catégorielles encodées :", cat_cols)

# --- OneHotEncoder (fit sur train)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_train = ohe.fit_transform(X_train[cat_cols])
ohe_test  = ohe.transform(X_test[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)

# --- Ajout des colonnes encodées
X_train_ohe = pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)
X_test_ohe  = pd.DataFrame(ohe_test,  columns=ohe_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=cat_cols+exclude_cols), X_train_ohe], axis=1)
X_test  = pd.concat([X_test.drop(columns=cat_cols+exclude_cols),  X_test_ohe], axis=1)

Colonnes catégorielles encodées : ['type_annonceur', 'typedebien', 'typedetransaction', 'annonce_exclusive', 'categorie_annonceur', 'typedebien_lite', 'TYP_IRIS_x', 'TYP_IRIS_y']


In [25]:
X_train.select_dtypes(include=['object', 'category']).columns

Index([], dtype='object')

In [26]:
X_train.select_dtypes(include='number').columns.to_list()

['etage',
 'surface',
 'surface_terrain',
 'nb_pieces',
 'mensualiteFinance',
 'balcon',
 'eau',
 'bain',
 'dpeC',
 'nb_etages',
 'places_parking',
 'cave',
 'annee_construction',
 'nb_toilettes',
 'ascenseur',
 'nb_logements_copro',
 'charges_copro',
 'logement_neuf',
 'INSEE_COM',
 'IRIS',
 'CODE_IRIS',
 'GRD_QUART',
 'UU2010',
 'REG',
 'DEP',
 'prix_m2_vente',
 'Latitude_scaled',
 'Longitude_scaled',
 'prix_bien_target_encoding',
 'prix_bien_target_encoding_scaled',
 'annee_2019',
 'annee_2020',
 'annee_2021',
 'annee_2022',
 'annee_2023',
 'expo_nord',
 'expo_sud',
 'expo_est',
 'expo_ouest',
 'expo_inconnue',
 'chauf_radiateur',
 'chauf_sol',
 'chauf_convecteur',
 'chauf_poele_bois',
 'chauf_pac',
 'chauf_clim_rev',
 'chauf_cheminee',
 'chauf_inconnu',
 'energie_gaz',
 'energie_elec',
 'energie_fioul',
 'energie_bois',
 'energie_inconnue',
 'dpe_A',
 'dpe_B',
 'dpe_C',
 'dpe_D',
 'dpe_E',
 'dpe_F',
 'dpe_G',
 'dpe_inconnu',
 'ges_A',
 'ges_B',
 'ges_C',
 'ges_D',
 'ges_E',
 'ges_F

In [27]:
X_train.describe()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,prix_bien_target_encoding_scaled,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,...,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
count,20569.000000,20569.000000,20569.00000,20569.000000,20569.0,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,2.056900e+04,2.056900e+04,20569.000000,20569.0,20569.0,20569.000000,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,...,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.0,20569.0,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000,20569.000000
mean,0.546162,104.613107,482.34325,4.435072,0.0,0.157762,0.275220,0.653070,206.918995,2.408576,1.856532,0.658612,1971.450824,1.453595,0.778647,29.022558,960.139683,0.145364,68193.253051,165.271185,6.819327e+08,6.819326e+06,68368.234090,44.0,68.0,2626.886578,3.573475e-14,-4.204393e-15,2.700713e+05,1.671947e-16,0.018329,0.235500,0.257426,0.342263,0.146483,0.033400,0.146288,0.066459,0.090816,0.770577,...,0.143566,0.212456,0.144343,0.053333,0.017259,0.323934,0.137829,0.124459,0.176090,0.201711,0.154359,0.074773,0.038650,0.092129,0.865283,0.138752,0.001993,0.0,1.0,0.495503,0.504497,0.000194,0.987846,0.011960,0.172493,0.405465,0.422043,0.747047,0.001507,0.036997,0.118333,0.082600,0.013515,0.495503,0.504497,0.000729,0.499344,0.499927,0.500073,0.499927
std,1.096705,49.332595,402.02828,1.774281,0.0,0.403882,0.509487,0.627869,86.844941,1.352000,1.084383,0.474187,37.826277,0.599168,0.415167,37.326728,711.542507,0.352476,107.171145,328.276630,1.071699e+06,1.071712e+04,269.831861,0.0,0.0,964.932838,1.000024e+00,1.000024e+00,1.264361e+05,1.000024e+00,0.134140,0.424321,0.437227,0.474479,0.353598,0.179683,0.353403,0.249089,0.287355,0.420472,...,0.350657,0.409055,0.351446,0.224702,0.130238,0.467987,0.344729,0.330113,0.380906,0.401288,0.361301,0.263031,0.192765,0.289215,0.341430,0.345696,0.044603,0.0,0.0,0.499992,0.499992,0.013944,0.109577,0.108707,0.377817,0.490994,0.493897,0.434715,0.038793,0.188760,0.323010,0.275283,0.115471,0.499992,0.499992,0.026995,0.500012,0.500012,0.500012,0.500012
min,0.000000,8.000000,1.00000,1.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1746.000000,0.000000,0.000000,0.000000,0.000000,0.000000,68001.000000,0.000000,6.800100e+08,6.800100e+06,68000.000000,44.0,68.0,15.590000,-2.097310e+00,-3.681208e+00,1.645000e+04,-2.005974e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0000

In [28]:
X_train.head()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,prix_bien_target_encoding_scaled,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,...,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,1,74,454.206,3,0,0,0,0,166.0,3,1,1,2005,1,1,12,1967.000,0,68135,0,681350000,6813500,68403,44,68,2814.86,-1.268439,1.437389,258272.800000,-0.093318,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0,122,1239.000,5,0,0,1,0,240.0,2,5,0,1974,2,1,7,70.534,0,68315,101,683150101,6831501,68401,44,68,3114.75,0.502046,-0.903823,306629.411765,0.289150,0.0,0.0,0.0,1.0,0.0,1,0,0,1,0,...,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,0,167,560.000,5,0,0,0,0,400.0,2,3,0,1970,1,1,4,4.800,0,68041,0,680410000,6804100,68000,44,68,772.46,0.412261,1.455061,298247.958333,0.222858,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0,129,1247.000,6,0,1,0,1,299.0,2,2,1,1984,2,1,14,988.800,0,68309,0,683090000,6830900,68115,44,68,3093.02,-0.839371,0.887967,451673.411765,1.436350,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
4,3,82,550.000,4,0,0,0,1,186.2,4,1,1,1799,1,1,80,2400.000,0,68224,201,682240201,6822401,68701,44,68,1195.12,-0.348123,-0.066819,161106.385017,-0.861839,0.0,0.0,1.0,0.0,0.0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [29]:
X_test.head()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,duree_int,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,loyer_m2_median_n7,nb_log_n7,taux_rendement_n7,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,prix_bien_target_encoding_scaled,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,...,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,0,96,110.074,5,0,0,1,0,153.0,2,1,1,1970,1,0,30,3500.0,0,75,68300,102,683000102,6830001,68701,44,68,7.91,2,0.0726,1202.08,-0.171263,0.200939,465000.000000,1.541754,0.0,0.0,0.0,1.0,0.0,0,...,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0,58,622.600,3,0,1,1,0,197.0,2,1,1,1914,1,0,15,240.0,0,198,68334,104,683340104,6833401,68402,44,68,9.51,76,0.0620,1836.21,0.043050,-1.846239,142705.915254,-1.007374,0.0,1.0,0.0,0.0,0.0,1,...,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,61,104.000,3,0,0,0,1,250.0,2,2,1,1954,1,1,14,0.0,0,317,68334,102,683340102,6833401,68402,44,68,9.51,76,0.0590,1938.52,-0.015535,-1.827338,142705.915254,-1.007374,0.0,1.0,0.0,0.0,0.0,0,...,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,0,74,569.600,4,0,0,0,1,216.0,4,1,1,1950,1,0,16,948.0,0,71,68224,1203,682241203,6822401,68701,44,68,9.22,388,0.1080,1027.03,-0.315539,-0.102853,161106.385017,-0.861839,0.0,0.0,0.0,0.0,1.0,0,...,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,4,166,344.000,7,0,0,1,1,156.0,4,1,1,1881,2,1,14,1389.0,0,7,68237,0,682370000,6823700,68000,44,68,7.96,4,0.0366,2349.40,1.638888,-0.760455,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0,...,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


écart dans le nombre de colonnes entre train et test ..

In [31]:
X_test.columns.to_list()

['etage',
 'surface',
 'surface_terrain',
 'nb_pieces',
 'mensualiteFinance',
 'balcon',
 'eau',
 'bain',
 'dpeC',
 'nb_etages',
 'places_parking',
 'cave',
 'annee_construction',
 'nb_toilettes',
 'ascenseur',
 'nb_logements_copro',
 'charges_copro',
 'logement_neuf',
 'duree_int',
 'INSEE_COM',
 'IRIS',
 'CODE_IRIS',
 'GRD_QUART',
 'UU2010',
 'REG',
 'DEP',
 'loyer_m2_median_n7',
 'nb_log_n7',
 'taux_rendement_n7',
 'prix_m2_vente',
 'Latitude_scaled',
 'Longitude_scaled',
 'prix_bien_target_encoding',
 'prix_bien_target_encoding_scaled',
 'annee_2019',
 'annee_2020',
 'annee_2021',
 'annee_2022',
 'annee_2023',
 'expo_nord',
 'expo_sud',
 'expo_est',
 'expo_ouest',
 'expo_inconnue',
 'chauf_radiateur',
 'chauf_sol',
 'chauf_convecteur',
 'chauf_poele_bois',
 'chauf_pac',
 'chauf_clim_rev',
 'chauf_cheminee',
 'chauf_inconnu',
 'energie_gaz',
 'energie_elec',
 'energie_fioul',
 'energie_bois',
 'energie_inconnue',
 'dpe_A',
 'dpe_B',
 'dpe_C',
 'dpe_D',
 'dpe_E',
 'dpe_F',
 'dpe_G',


In [32]:
X_train.columns.to_list()

['etage',
 'surface',
 'surface_terrain',
 'nb_pieces',
 'mensualiteFinance',
 'balcon',
 'eau',
 'bain',
 'dpeC',
 'nb_etages',
 'places_parking',
 'cave',
 'annee_construction',
 'nb_toilettes',
 'ascenseur',
 'nb_logements_copro',
 'charges_copro',
 'logement_neuf',
 'INSEE_COM',
 'IRIS',
 'CODE_IRIS',
 'GRD_QUART',
 'UU2010',
 'REG',
 'DEP',
 'prix_m2_vente',
 'Latitude_scaled',
 'Longitude_scaled',
 'prix_bien_target_encoding',
 'prix_bien_target_encoding_scaled',
 'annee_2019',
 'annee_2020',
 'annee_2021',
 'annee_2022',
 'annee_2023',
 'expo_nord',
 'expo_sud',
 'expo_est',
 'expo_ouest',
 'expo_inconnue',
 'chauf_radiateur',
 'chauf_sol',
 'chauf_convecteur',
 'chauf_poele_bois',
 'chauf_pac',
 'chauf_clim_rev',
 'chauf_cheminee',
 'chauf_inconnu',
 'energie_gaz',
 'energie_elec',
 'energie_fioul',
 'energie_bois',
 'energie_inconnue',
 'dpe_A',
 'dpe_B',
 'dpe_C',
 'dpe_D',
 'dpe_E',
 'dpe_F',
 'dpe_G',
 'dpe_inconnu',
 'ges_A',
 'ges_B',
 'ges_C',
 'ges_D',
 'ges_E',
 'ges_F